# Instrumented Evidence-Summarization Pipeline

This notebook demonstrates how the Briefcase AI SDK instruments an existing LLM summarization pipeline. Every call is captured via the `@capture` decorator and persisted as a `DecisionSnapshot` — recording inputs, outputs, model parameters, execution time, hardware metadata, and data versioning.

**What you'll see:**
- `@capture` decorator for automatic function-level instrumentation
- `DecisionSnapshot` capturing the full decision context for every LLM call
- `briefcase_workflow` for automatic W3C traceparent propagation across agents
- `briefcase.setup()` for centralized SDK configuration
- `detect_hardware()` for reproducibility metadata
- `SqliteBackend` for local persistence — no external services, fully air-gappable
- Multi-model switching via `ModelParameters` (GPT-4o and Claude Sonnet side by side)
- Content-hash data versioning that maps directly to lakeFS commit SHAs in production

> All cells run fully offline using `MockLLMProvider`. No API keys needed.

In [ ]:
import sys, os
os.chdir(os.path.join(os.path.dirname(os.path.abspath(".")), ""))
# Ensure project root is importable
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

import json
import hashlib
from pathlib import Path

import briefcase
from briefcase.correlation import briefcase_workflow
from briefcase.decorators import capture
from briefcase.hardware import detect_hardware
from src.mock_llm import MockLLMProvider
from src.pipeline import summarize_report
from src.config import storage, config

print(f"Briefcase AI SDK v{briefcase.__version__}")
print(f"Storage backend: SqliteBackend (decisions.db)")
print(f"Configuration: centralized via briefcase.setup()")

## 1. Load a Police Report

We use 5 synthetic police reports as toy data. Each report is a realistic incident narrative — burglary, DUI, assault, retail theft — with structured fields that give evaluation guardrails something to work with.

No DUA required. All data is fictional.

In [ ]:
report_text = Path("data/police_reports/report_001.txt").read_text()

print(f"Report: report_001 (Residential Burglary)")
print(f"Length: {len(report_text)} characters")
print(f"{'─' * 60}")
# Show the header and first paragraph
for line in report_text.split('\n')[:12]:
    print(line)
print("...")

## 2. Run the Instrumented Pipeline

`summarize_report()` is decorated with `@capture(decision_type="evidence_summarization")` which auto-records inputs, outputs, timing, and errors. Inside, it builds a `DecisionSnapshot` for persistence and captures:

| Field | What it records |
|-------|----------------|
| `Input("document", ...)` | The source police report text |
| `Input("query", ...)` | The summarization prompt |
| `ModelParameters` | Model name, provider, temperature, max_tokens |
| `Output("summary", ...).with_confidence()` | Generated summary + model confidence |
| `execution_time_ms` | Wall-clock latency of the LLM call |
| `data_version` tag | SHA-256 content hash of the source document |
| `hardware_type` / `hardware_name` tags | Execution environment for reproducibility |
| `data_ref_uri` tag | URI pointer to the source file |

The `@capture` decorator handles the observability layer while `DecisionSnapshot` handles persistence — two complementary layers.

In [ ]:
# Summarize with GPT-4o (mocked)
gpt4o = MockLLMProvider(model="gpt-4o", provider="openai", simulate_latency=False)
result = await summarize_report("report_001", report_text, llm=gpt4o)

print(f"Model:       {result['model']} ({result['provider']})")
print(f"Confidence:  {result['confidence']}")
print(f"Snapshot ID: {result['snapshot_id']}")
print(f"Data Version:{result['data_version']}")
print(f"Hardware:    {result['hardware_type']} ({result['hardware_name']})")
print(f"Tokens:      {result['token_usage']}")
print(f"Latency:     {result['latency_ms']:.1f}ms")
print(f"\n{'─' * 60}")
print(f"SUMMARY:\n")
print(result["summary"])

## 3. Inspect the Stored Decision

Every decision is persisted to `SqliteBackend`. We can load it back and inspect every field — this is what makes decisions auditable and replayable.

In [ ]:
# Load the decision back from storage
loaded = storage.load_decision(result["snapshot_id"])

print(f"Function:    {loaded.function_name}")
print(f"Exec time:   {loaded.execution_time_ms:.1f}ms")
print(f"Tags:        {loaded.tags}")
print(f"\nHardware metadata (for reproducibility):")
print(f"  Type: {loaded.tags.get('hardware_type', 'N/A')}")
print(f"  Name: {loaded.tags.get('hardware_name', 'N/A')}")

print(f"\nInputs ({len(loaded.inputs)}):")
for inp in loaded.inputs:
    val_preview = inp.value[:80] + "..." if len(inp.value) > 80 else inp.value
    print(f"  - {inp.name} ({inp.data_type}): {val_preview}")

print(f"\nOutputs ({len(loaded.outputs)}):")
for out in loaded.outputs:
    val_preview = out.value[:80] + "..." if len(out.value) > 80 else out.value
    print(f"  - {out.name} ({out.data_type}): {val_preview}")
    print(f"    confidence: {out.confidence}")

## 4. Multi-Model Comparison

The pipeline supports switching models via `ModelParameters`. Same report, different model — both decisions are tracked independently.

In [ ]:
# Now summarize with Claude Sonnet (mocked)
claude = MockLLMProvider(model="claude-sonnet", provider="anthropic", simulate_latency=False)
result_claude = await summarize_report("report_001", report_text, llm=claude)

print(f"{'GPT-4o':>14} | {'Claude Sonnet':>14}")
print(f"{'─' * 14}-+-{'─' * 14}")
print(f"{'Confidence:':<14} {result['confidence']:>6.2f}   | {result_claude['confidence']:>6.2f}")
print(f"{'Tokens:':<14} {result['token_usage']['total_tokens']:>6}   | {result_claude['token_usage']['total_tokens']:>6}")

# Show how the summaries differ
print(f"\n--- GPT-4o (first 200 chars) ---")
print(result["summary"][:200] + "...")
print(f"\n--- Claude Sonnet (first 200 chars) ---")
print(result_claude["summary"][:200] + "...")

## 5. Process All Reports

Scale to the full report set. Each report is summarized by both models, producing 10 `DecisionSnapshot` records — all traceable to their source document version.

In [ ]:
report_ids = ["report_001", "report_002", "report_003", "report_004", "report_005"]
models = [
    MockLLMProvider(model="gpt-4o", provider="openai", simulate_latency=False),
    MockLLMProvider(model="claude-sonnet", provider="anthropic", simulate_latency=False),
]

all_results = []
for llm in models:
    for rid in report_ids:
        text = Path(f"data/police_reports/{rid}.txt").read_text()
        r = await summarize_report(rid, text, llm=llm)
        all_results.append(r)

# Summary table
print(f"{'Report':<12} {'Model':<16} {'Conf':>5} {'Tokens':>7} {'Data Version':<14}")
print("─" * 58)
for r in all_results:
    print(f"{r['snapshot_id'][:8]}...  {r['model']:<16} {r['confidence']:>5.2f} {r['token_usage']['total_tokens']:>7} {r['data_version']}")

print(f"\nTotal DecisionSnapshots stored: {len(all_results)}")

## 6. Data Versioning

Every snapshot includes a content-hash `data_version` tag — a SHA-256 fingerprint of the source document. This is the same pattern used with lakeFS commit SHAs in production.

If the source document changes, the hash changes, and the lineage is traceable.

In [ ]:
# Show that different reports produce different version hashes
print("Content-hash versioning (simulates lakeFS commit SHA):\n")
for rid in report_ids:
    text = Path(f"data/police_reports/{rid}.txt").read_text()
    content_hash = hashlib.sha256(text.encode()).hexdigest()[:12]
    print(f"  {rid}.txt -> {content_hash}  ({len(text)} chars)")

# Show that the amended report has a different hash
text_orig = Path("data/police_reports/report_001.txt").read_text()
text_amended = Path("data/police_reports/report_001_amended.txt").read_text()
hash_orig = hashlib.sha256(text_orig.encode()).hexdigest()[:12]
hash_amended = hashlib.sha256(text_amended.encode()).hexdigest()[:12]

print(f"\n  report_001.txt         -> {hash_orig}")
print(f"  report_001_amended.txt -> {hash_amended}")
print(f"\n  Same document? {hash_orig == hash_amended}  (amended version adds detective addendum)")

## Key Takeaways

- **`@capture` decorator**: Auto-records function inputs, outputs, timing, and errors — the observability layer
- **`DecisionSnapshot` persistence**: Every decision is stored with full audit trail via `SqliteBackend`
- **Centralized config**: `briefcase.setup()` configures storage, event bus, and routing in one place
- **Hardware metadata**: `detect_hardware()` tags every decision with the execution environment for reproducibility
- **Multi-model support**: Switch between GPT-4o and Claude Sonnet via `ModelParameters`
- **Local-first storage**: `SqliteBackend` works offline, air-gapped, Azure Gov Cloud compatible
- **Data versioning**: Content-hash tags link every decision to its exact source document version

**Next:** See [02_evaluation_and_guardrails.ipynb](./02_evaluation_and_guardrails.ipynb) for structured evaluation using 3 distinct `GuardrailEnv` implementations.